# Document Classification by Conflict Type

This notebook classifies MIMIC-III clinical documents into the 4 specialized conflict types from the `ht/multi-doctor-agents` branch:

1. **Temporality** - Allergies, constitutional patient characteristics, immutable facts
2. **Clinical History** - Surgical files, transplant records, implants
3. **Biomarker** - Lab values, vital signs, physiological measurements
4. **Pre/Post Care** - Admission/discharge documents, care transitions

Uses embedding similarity (sentence-transformers) for multi-label classification with **per-type thresholds**.

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from collections import Counter, defaultdict
from pathlib import Path
import json

In [ ]:
# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Load data
data_path = Path("..") / "data" / "mimic-iii-verifact-bhc.parquet"
df = pd.read_parquet(data_path)
print(f"Loaded {len(df)} documents from {df.subject_id.nunique()} patients")

## Conflict Type Descriptions

These descriptions are derived from the specialized doctor agent prompts in `ht/multi-doctor-agents` branch.

In [ ]:
CONFLICT_DESCRIPTIONS = {
    "temporality": """Long-term follow-up documents for allergies or constitutional patient characteristics.
        Immutable characteristics like blood type, genetic markers, birth date.
        Allergy documentation, chronic irreversible conditions like Type 1 diabetes.
        Constitutional metrics like adult height. Patient characteristics that should remain constant.""",
    "clinical_history": """Surgical files, transplant records, pre and post operative documents.
        Surgical history, organ removal, appendectomy, cholecystectomy.
        Transplant status, immunosuppression, implanted medical devices, pacemaker.
        Anatomical alterations from prior procedures.""",
    "biomarker": """Laboratory values, vital signs, physiological measurements.
        Hemoglobin, glucose, troponin, creatinine, WBC counts.
        Post-operative biomarker trajectories, organ function markers.
        Immunological markers, procedural marker changes.""",
    "pre_post_care": """Admission and discharge documents, care transition points.
        Chief complaint, admission diagnosis, discharge summary.
        Medication reconciliation, functional status changes.
        Diagnosis changes between admission and discharge.""",
}

# Per-type thresholds
# Those thresholds capture documents that "mention" the topic (at least one sentence)
PER_TYPE_THRESHOLDS = {
    "temporality": 0.09,
    "clinical_history": 0.16,
    "biomarker": 0.21,
    "pre_post_care": 0.23,
}

print("Per-type thresholds:")
for ct, thresh in PER_TYPE_THRESHOLDS.items():
    print(f"  {ct}: {thresh}")

In [ ]:
# Embed conflict type descriptions
desc_embeddings = {k: model.encode(v) for k, v in CONFLICT_DESCRIPTIONS.items()}
print("Embedded conflict type descriptions")

## Multi-Label Document Classification

Each document can belong to multiple conflict types if similarity exceeds the per-type threshold.

In [ ]:
def classify_document(text, thresholds=PER_TYPE_THRESHOLDS):
    """
    Classify a document into conflict types using embedding similarity.

    Args:
        text: Document text
        thresholds: Dict of per-type minimum similarity scores

    Returns:
        dict with 'matching_types', 'scores', 'best_match'
    """
    doc_emb = model.encode(text[:1000])

    # Compute cosine similarity with each conflict type
    similarities = {}
    for k, v in desc_embeddings.items():
        cos_sim = np.dot(doc_emb, v) / (np.linalg.norm(doc_emb) * np.linalg.norm(v))
        similarities[k] = float(cos_sim)

    # Multi-labeling
    matching_types = [k for k, v in similarities.items() if v >= thresholds[k]]
    best_match = max(similarities, key=similarities.get)

    return {"matching_types": matching_types, "scores": similarities, "best_match": best_match}

In [ ]:
# Classify all documents
results = []
for idx, row in df.iterrows():
    classification = classify_document(row["text"])
    classification["category"] = row["category"]
    classification["row_id"] = row["row_id"]
    classification["subject_id"] = row["subject_id"]
    results.append(classification)

    if (idx + 1) % 1000 == 0:
        print(f"Processed {idx + 1}/{len(df)} documents...")

print(f"\nClassified {len(results)} documents")

In [ ]:
results_df = pd.DataFrame(results)

# Add binary columns for each conflict type
for ct in CONFLICT_DESCRIPTIONS.keys():
    results_df[f"is_{ct}"] = results_df["matching_types"].apply(lambda x: ct in x)

results_df["num_types"] = results_df["matching_types"].apply(len)
results_df.head()

## Results Summary

In [ ]:
print("=" * 70)
print("MULTI-LABEL CLASSIFICATION (per-type thresholds)")
print("=" * 70)

print("\nDocuments per conflict type (can overlap):")
for ct in ["temporality", "clinical_history", "biomarker", "pre_post_care"]:
    count = results_df[f"is_{ct}"].sum()
    thresh = PER_TYPE_THRESHOLDS[ct]
    print(f"  {ct:20}: {count:5} docs ({count/len(df)*100:5.1f}%)  [threshold={thresh}]")

print("\nNumber of conflict types per document:")
for n, count in results_df["num_types"].value_counts().sort_index().items():
    print(f"  {n} types: {count:5} docs ({count/len(df)*100:.1f}%)")

In [ ]:
# Cross-tabulation: MIMIC category vs Conflict types
print("BREAKDOWN BY MIMIC CATEGORY (multi-label counts)")
print("=" * 70)

cross_tab = (
    results_df.groupby("category")
    .agg(
        {
            "is_temporality": "sum",
            "is_clinical_history": "sum",
            "is_biomarker": "sum",
            "is_pre_post_care": "sum",
        }
    )
    .astype(int)
)

cross_tab["total_docs"] = results_df.groupby("category").size()
cross_tab.columns = ["temporality", "clinical_history", "biomarker", "pre_post_care", "total_docs"]
cross_tab

In [ ]:
# Percentage view
pct_tab = cross_tab.copy()
for col in ["temporality", "clinical_history", "biomarker", "pre_post_care"]:
    pct_tab[col] = (cross_tab[col] / cross_tab["total_docs"] * 100).round(1)

print("Percentage of documents in each category matching each conflict type:")
pct_tab

## Save Document Lists per Conflict Type

In [ ]:
# Create dict of document row_ids for each conflict type
docs_per_conflict_type = {}

for ct in ["temporality", "clinical_history", "biomarker", "pre_post_care"]:
    matching_docs = results_df[results_df[f"is_{ct}"]]["row_id"].tolist()
    docs_per_conflict_type[ct] = matching_docs
    print(f"{ct}: {len(matching_docs)} documents")

# Save to JSON file
output_path = Path("..") / "data" / "documents_per_conflict_type.json"
with open(output_path, "w") as f:
    json.dump(docs_per_conflict_type, f, indent=2)
print(f"\nSaved document lists to {output_path}")

In [ ]:
# Also save as separate CSV files for convenience
output_dir = Path("..") / "data" / "conflict_type_documents"
output_dir.mkdir(exist_ok=True)

for ct in ["temporality", "clinical_history", "biomarker", "pre_post_care"]:
    # Get full document info for this conflict type
    ct_docs = results_df[results_df[f"is_{ct}"]][["row_id", "subject_id", "category"]].copy()
    ct_docs["score"] = results_df[results_df[f"is_{ct}"]]["scores"].apply(lambda x: x[ct])
    ct_docs = ct_docs.sort_values("score", ascending=False)

    # Save to CSV
    csv_path = output_dir / f"{ct}_documents.csv"
    ct_docs.to_csv(csv_path, index=False)
    print(f"Saved {len(ct_docs)} docs to {csv_path}")

## Recommended Category Mapping

Based on the analysis, these MIMIC categories are best suited for each conflict type:

In [ ]:
# Find best categories for each conflict type (>50% match rate)
MIN_MATCH_RATE = 50

RECOMMENDED_MAPPING = {}
for ct in ["temporality", "clinical_history", "biomarker", "pre_post_care"]:
    good_cats = pct_tab[pct_tab[ct] >= MIN_MATCH_RATE].index.tolist()
    RECOMMENDED_MAPPING[ct] = good_cats

print("RECOMMENDED CATEGORY MAPPING (>50% match rate):")
print("-" * 50)
for ct, cats in RECOMMENDED_MAPPING.items():
    print(f"{ct}:")
    if cats:
        for cat in cats:
            print(f"  - {cat} ({int(cross_tab.loc[cat, ct])} docs, {pct_tab.loc[cat, ct]}%)")
    else:
        print(f"  - No categories with >50% match rate")
    print()

In [ ]:
# Export the mapping for use in the pipeline
print("Code to use in pipeline:")
print("-" * 50)
print("CONFLICT_TO_CATEGORIES = {")
for ct, cats in RECOMMENDED_MAPPING.items():
    print(f'    "{ct}": {cats},')
print("}")

## Save Full Classification Results

In [ ]:
# Save full classification results
output_path = Path("..") / "data" / "document_conflict_classification.parquet"

# Prepare export DataFrame
export_df = results_df[
    [
        "row_id",
        "subject_id",
        "category",
        "best_match",
        "num_types",
        "is_temporality",
        "is_clinical_history",
        "is_biomarker",
        "is_pre_post_care",
    ]
].copy()

# Add scores as separate columns
for ct in CONFLICT_DESCRIPTIONS.keys():
    export_df[f"score_{ct}"] = results_df["scores"].apply(lambda x: x[ct])

export_df.to_parquet(output_path, index=False)
print(f"Saved full classification results to {output_path}")
print(f"\nColumns: {list(export_df.columns)}")